# OpenFedLLM Federated Learning Training - Google Colab Version

This notebook runs federated learning training on Google Colab, without requiring GPT API evaluation.

## Instructions
1. When running in Colab, make sure a GPU runtime is selected (Runtime -> Change runtime type -> GPU)
2. Run all cells in order
3. After training finishes, you can view the training loss curve

## Step 1: Check and Install Dependencies (Smart Install)

Colab comes with many packages preinstalled, so we only install packages that are missing or have mismatched versions.

In [ ]:
# First check which packages are already installed and which need to be installed
import importlib
import subprocess
import sys

# Packages to check and their required versions
required_packages = {
    'transformers': '4.31.0',
    'peft': '0.4.0',
    'trl': '0.7.2',
    'bitsandbytes': '0.40.2',
    'accelerate': '0.21.0',
    'datasets': '2.13.0',
    'torch': None,  # Only check that it's installed; version is determined by Colab
    'tqdm': None,  # Usually already installed
    'numpy': None,  # Usually already installed
}

print("🔍 Checking installed packages...")
packages_to_install = []

for package, required_version in required_packages.items():
    try:
        module = importlib.import_module(package)
        installed_version = getattr(module, '__version__', 'unknown')
        
        if required_version:
            # Check whether the version matches
            if installed_version != required_version:
                print(f"  ⚠️  {package}: {installed_version} (requires {required_version})")
                packages_to_install.append(f"{package}=={required_version}")
            else:
                print(f"  ✅ {package}: {installed_version} (installed, correct version)")
        else:
            print(f"  ✅ {package}: {installed_version} (installed)")
    except ImportError:
        print(f"  ❌ {package}: not installed")
        if required_version:
            packages_to_install.append(f"{package}=={required_version}")
        else:
            packages_to_install.append(package)

print("\n" + "="*60)
if packages_to_install:
    print(f"{len(packages_to_install)} package(s) need to be installed:")
    for pkg in packages_to_install:
        print(f"  - {pkg}")
else:
    print("✅ All required packages are installed with the correct versions!")
    print("You can skip the installation step")

In [ ]:
# Only install packages that are missing or have mismatched versions
if packages_to_install:
    print(f"📦 Installing {len(packages_to_install)} package(s)...")
    print("\n" + "="*60)
    
    # Install in a single batch (more efficient)
    install_cmd = f"pip install -q {' '.join(packages_to_install)}"
    print(f"Running command: {install_cmd}")
    
    result = subprocess.run(
        install_cmd.split(),
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("\n✅ Dependencies installed!")
    else:
        print(f"\n⚠️ There may have been warnings during installation, but you can usually continue")
        if result.stderr:
            print(f"Error message: {result.stderr[:500]}")
else:
    print("✅ All packages are installed, skipping installation step")

# Verify installation
print("\n🔍 Verifying installation...")
failed_packages = []
for package in required_packages.keys():
    try:
        importlib.import_module(package)
        print(f"  ✅ {package}")
    except ImportError:
        print(f"  ❌ {package} failed to import")
        failed_packages.append(package)

if failed_packages:
    print(f"\n⚠️ The following packages failed to import: {failed_packages}")
    print("Please install them manually: !pip install " + " ".join(failed_packages))
else:
    print("\n✅ All packages verified!")

## Step 2: Clone the Repository

In [ ]:
import os

# Clone the repository (if not already present)
repo_url = 'https://github.com/GuangLun2000/OpenFedLLM-Attack.git'
repo_name = 'OpenFedLLM-Attack'

if not os.path.exists(repo_name):
    print(f"Cloning repository: {repo_url}")
    !git clone --recursive --shallow-submodules {repo_url} {repo_name}
    print("✅ Repository cloned!")
else:
    print("✅ Repository already exists, skipping clone")

# Change into the project directory
os.chdir(repo_name)
print(f"Current working directory: {os.getcwd()}")

# Verify that key files exist
required_files = ['main_sft.py', 'config.py', 'utils', 'federated_learning']
for file in required_files:
    if os.path.exists(file):
        print(f"  ✅ {file} exists")
    else:
        print(f"  ❌ {file} does not exist!")

## Step 3: Set Up Environment Variables

In [ ]:
import os
import sys

# Set PYTHONPATH
fingpt_dir = 'evaluation/FinGPT'
if os.path.exists(fingpt_dir):
    sys.path.insert(0, os.path.abspath(fingpt_dir))

# Add the current directory to the Python path
sys.path.insert(0, os.getcwd())

print("✅ Environment variables set!")
print(f"Python path: {sys.path[:3]}")

## Step 4: Debug and Verify the Environment

In [ ]:
# Verify that all required modules can be imported
import sys
import os

print("🔍 Environment check:")
print("\n1. Checking Python path:")
print(f"   Current directory: {os.getcwd()}")
print(f"   Python path contains it: {os.getcwd() in sys.path}")

print("\n2. Checking key files:")
key_files = ['main_sft.py', 'config.py', 'utils/__init__.py', 'federated_learning/__init__.py']
for file in key_files:
    exists = os.path.exists(file)
    status = "✅" if exists else "❌"
    print(f"   {status} {file}")

print("\n3. Trying to import key modules:")
try:
    from utils import *
    print("   ✅ utils module imported successfully")
except Exception as e:
    print(f"   ❌ utils module failed to import: {e}")

try:
    from federated_learning import *
    print("   ✅ federated_learning module imported successfully")
except Exception as e:
    print(f"   ❌ federated_learning module failed to import: {e}")

try:
    from config import get_config
    print("   ✅ config module imported successfully")
except Exception as e:
    print(f"   ❌ config module failed to import: {e}")

print("\n4. Checking dependencies:")
packages = ['transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'datasets', 'torch']
for pkg in packages:
    try:
        __import__(pkg)
        print(f"   ✅ {pkg}")
    except ImportError:
        print(f"   ❌ {pkg} not installed")

print("\n✅ Environment check complete!")

## Step 5: Check GPU and CUDA

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Warning: No GPU detected. Make sure a GPU runtime is selected in Colab!")
    print("   How to set it: Runtime -> Change runtime type -> Hardware accelerator -> GPU")

## Step 6: Configure Training Parameters (Small-Scale Configuration Suited to Colab)

In [ ]:
# ========== Training Parameter Configuration ==========
# These parameters have been tuned to fit Colab's resource limits

training_config = {
    # Model configuration
    'model_name_or_path': 'meta-llama/Llama-2-7b-hf',  # Base model
    'use_peft': True,  # Use LoRA (required)
    'peft_lora_r': 16,  # LoRA rank (lowered to save GPU memory)
    'peft_lora_alpha': 32,  # LoRA alpha
    'load_in_8bit': True,  # 8-bit quantization (required, saves GPU memory)
    
    # Dataset configuration
    'dataset_name': 'vicgalle/alpaca-gpt4',  # Dataset
    'dataset_sample': 5000,  # Use 5000 samples (reduced data size)
    'template': 'alpaca',  # Template
    
    # Federated learning configuration
    'fed_alg': 'fedavg',  # Federated learning algorithm
    'num_clients': 5,  # Number of clients (reduced)
    'sample_clients': 2,  # Number of clients sampled per round
    'num_rounds': 10,  # Number of training rounds (reduced for quick testing)
    
    # Training configuration
    'max_steps': 5,  # Training steps per client per round (reduced)
    'batch_size': 4,  # Batch size (lowered to save GPU memory)
    'gradient_accumulation_steps': 4,  # Gradient accumulation (keeps the effective batch size)
    'learning_rate': 2e-5,  # Learning rate
    'seq_length': 256,  # Sequence length (lowered to save GPU memory)
    
    # Output configuration
    'output_dir': './output_colab',  # Output directory
    'save_model_freq': 5,  # Save the model every 5 rounds
    'logging_steps': 10,  # Logging steps
    'seed': 2023,  # Random seed
}

print("✅ Training parameters configured!")
print("\nConfiguration summary:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

## Step 7: Prepare Command-Line Arguments

In [ ]:
# Build the list of command-line arguments
args_list = ['main_sft.py']  # Script name

# Add all arguments
for key, value in training_config.items():
    if isinstance(value, bool):
        if value:
            args_list.append(f"--{key}")
    else:
        args_list.append(f"--{key}")
        args_list.append(str(value))

# Add federated-learning-specific arguments (if not already present)
if '--save_model_freq' not in args_list:
    args_list.extend([
        '--save_model_freq', str(training_config['save_model_freq']),
    ])

print("✅ Command-line arguments prepared!")
print(f"\nCommand to be executed:")
print(f"python {' '.join(args_list)}")
print(f"\nNumber of arguments: {len(args_list)}")

## Step 8: Start Training

In [ ]:
# Build the full command string
cmd_args = args_list[1:]  # Remove the script name
cmd_str = ' '.join([str(arg) for arg in cmd_args])

print("🚀 Starting federated learning training...")
print("⏰ Estimated time: 30 minutes to 2 hours, depending on GPU performance")
print(f"\nRunning command: python main_sft.py {cmd_str[:100]}...")
print("\n" + "="*60)
print("\nRun the training command in the cell below (using !python)")
print("\nOr uncomment the code below to run it automatically:")

In [ ]:
# Run the training command
# Note: In Colab, running a Python script with a ! command is the most reliable method

import sys
import subprocess
import os

# Make sure we're in the correct directory
if not os.path.exists('main_sft.py'):
    print("❌ Error: main_sft.py not found. Make sure you're in the project root directory")
    print(f"Current directory: {os.getcwd()}")
else:
    print("Running training...")
    print(f"Working directory: {os.getcwd()}")
    print(f"Python executable: {sys.executable}")
    print("\n" + "="*60 + "\n")

    try:
        # Run with subprocess and capture the output
        result = subprocess.run(
            [sys.executable, 'main_sft.py'] + cmd_args,
            check=False,
            capture_output=True,
            text=True,
            cwd=os.getcwd()
        )
        
        # Show the output
        if result.stdout:
            print("Standard output:")
            print(result.stdout)
        
        if result.stderr:
            print("\nError output:")
            print(result.stderr)
        
        print("\n" + "="*60)
        if result.returncode == 0:
            print("✅ Training complete!")
        else:
            print(f"❌ An error occurred during training, return code: {result.returncode}")
            print("\n💡 Debugging tips:")
            print("  1. Check the error output above")
            print("  2. Make sure all dependencies are installed")
            print("  3. Check that the GPU is available")
            print("  4. Try the direct execution method below")
            print("\n💡 Or run the following command manually:")
            print(f"!python main_sft.py {cmd_str}")
    except Exception as e:
        print(f"❌ Execution error: {e}")
        import traceback
        traceback.print_exc()
        print("\n💡 Please run the following command manually:")
        print(f"!python main_sft.py {cmd_str}")

## Step 7 (Alternative): Run the Command Directly (If the Method Above Fails)

In [ ]:
# Method 2: Run directly with a ! command (recommended, more reliable)
# Use this method if the subprocess method above fails

print("💡 If the method above fails, use the direct execution method below:")
print("\n" + "="*60)
print("Command to run (copy into a new cell and run it):")
print("\n" + "="*60)
print(f"!python main_sft.py {cmd_str}")
print("\n" + "="*60)
print("\nOr run the code below directly:")
print("\n(uncomment the code below)")

# Uncomment the line below to run directly
# !python main_sft.py {cmd_str}

In [ ]:
# Method 2: Run directly with a ! command (the most reliable method)
# In Colab, running directly with a ! command is the most reliable method

import os

# Make sure we're in the correct directory
if os.path.exists('main_sft.py'):
    print("🚀 Preparing to run the training command...")
    print(f"Working directory: {os.getcwd()}")
    print("\n" + "="*60)
    print("Copy the following command into a new cell and run it:")
    print("\n" + "="*60)
    print(f"!python main_sft.py {cmd_str}")
    print("\n" + "="*60)
    print("\nOr run it directly in this cell (uncomment the code below):")
    print("\nNote: In Jupyter/Colab, ! commands need to be run in a separate cell")
    
    # The code below shows the command when run, but actually executing it requires the user to create a new cell manually
    # or use get_ipython().run_cell_magic()
    try:
        from IPython import get_ipython
        ipython = get_ipython()
        if ipython:
            print("\n✅ IPython environment detected, can run directly")
            print("Uncomment the code below to run it:")
            # Uncomment the line below to run
            # ipython.run_cell_magic('bash', '', f'python main_sft.py {cmd_str}')
    except:
        pass
else:
    print("❌ Error: main_sft.py not found")
    print(f"Current directory: {os.getcwd()}")
    print("Make sure you're in the project root directory")

## Step 8: Visualize Training Loss (No API Required)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Load the training loss
loss_file = os.path.join(training_config['output_dir'], 'training_loss.npy')

if os.path.exists(loss_file):
    training_loss = np.load(loss_file)
    
    print(f"Training loss shape: {training_loss.shape}")
    print(f"Number of clients: {training_loss.shape[0]}")
    print(f"Number of training rounds: {training_loss.shape[1]}")
    
    # Plot the loss curves
    plt.figure(figsize=(12, 6))
    
    # Plot the loss for each client
    for client_id in range(training_loss.shape[0]):
        client_loss = training_loss[client_id]
        # Filter out -1 (rounds the client did not participate in)
        valid_rounds = np.where(client_loss >= 0)[0]
        if len(valid_rounds) > 0:
            plt.plot(valid_rounds + 1, client_loss[valid_rounds], 
                    alpha=0.6, label=f'Client {client_id}', linewidth=1.5)
    
    # Compute and plot the average loss
    avg_loss = []
    for round_id in range(training_loss.shape[1]):
        round_losses = training_loss[:, round_id]
        valid_losses = round_losses[round_losses >= 0]
        if len(valid_losses) > 0:
            avg_loss.append(np.mean(valid_losses))
        else:
            avg_loss.append(np.nan)
    
    valid_avg_rounds = [i+1 for i, v in enumerate(avg_loss) if not np.isnan(v)]
    valid_avg_loss = [v for v in avg_loss if not np.isnan(v)]
    
    if len(valid_avg_rounds) > 0:
        plt.plot(valid_avg_rounds, valid_avg_loss, 
                'k-', linewidth=3, label='Average Loss', marker='o', markersize=6)
    
    plt.xlabel('Training Round', fontsize=12)
    plt.ylabel('Training Loss', fontsize=12)
    plt.title('Federated Learning Training Loss', fontsize=14, fontweight='bold')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\n📊 Training statistics:")
    print(f"  Initial average loss: {valid_avg_loss[0]:.4f}")
    print(f"  Final average loss: {valid_avg_loss[-1]:.4f}")
    print(f"  Loss decrease: {valid_avg_loss[0] - valid_avg_loss[-1]:.4f}")
    print(f"  Loss decrease percentage: {(valid_avg_loss[0] - valid_avg_loss[-1]) / valid_avg_loss[0] * 100:.2f}%")
    
else:
    print(f"❌ Training loss file not found: {loss_file}")
    print("   Make sure training has finished and the loss file was saved")

## Step 10: Check Output Files

In [ ]:
import os
import json

output_dir = training_config['output_dir']

if os.path.exists(output_dir):
    print(f"✅ Output directory exists: {output_dir}")
    print("\n📁 Output files:")
    
    for item in os.listdir(output_dir):
        item_path = os.path.join(output_dir, item)
        if os.path.isdir(item_path):
            print(f"  📂 {item}/ (directory)")
        else:
            size = os.path.getsize(item_path)
            if size < 1024:
                size_str = f"{size} B"
            elif size < 1024 * 1024:
                size_str = f"{size / 1024:.2f} KB"
            else:
                size_str = f"{size / (1024 * 1024):.2f} MB"
            print(f"  📄 {item} ({size_str})")
    
    # Read and display the configuration
    config_file = os.path.join(output_dir, 'args.json')
    if os.path.exists(config_file):
        print("\n📋 Training configuration:")
        with open(config_file, 'r') as f:
            config = json.load(f)
        print(json.dumps(config, indent=2, ensure_ascii=False))
else:
    print(f"❌ Output directory does not exist: {output_dir}")

## 💡 Usage Tips

### If you run into problems:

1. **Out of GPU memory (OOM)**:
   - Further reduce `batch_size` (to 2 or 1)
   - Reduce `seq_length` (to 128)
   - Reduce `peft_lora_r` (to 8)
   - Reduce `dataset_sample` (to 2000)

2. **Training is too slow**:
   - Reduce `num_rounds` (to 5)
   - Reduce `max_steps` (to 3)
   - Reduce `dataset_sample` (to 2000)

3. **Model download fails**:
   - Make sure you're logged in to HuggingFace: `huggingface-cli login`
   - Or use the path to a locally downloaded model

4. **You want more thorough training**:
   - Increase `num_rounds` (to 20-50)
   - Increase `max_steps` (to 10)
   - Increase `dataset_sample` (to 10000)
   - Increase `num_clients` (to 10)

### Next steps:
- After training finishes, the model is saved in the `output_colab/checkpoint-*` directory
- You can use the trained model for inference
- For evaluation, you can run the evaluation scripts (requires an API key)